In [1]:
# Cell 1: Install dependencies
!pip install kagglehub ultralytics pyyaml

INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 169.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 21.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 146.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [ultralytics] [ultralytics]thop]


In [2]:
import kagglehub
import os
import shutil

# Download dataset from Kaggle
print("Downloading NepaliHTR dataset from Kaggle...")
cache_path = kagglehub.dataset_download("sweekardahal/nepali-handwritten-images-for-text-detection")
print(f"Downloaded to cache: {cache_path}")

# Copy to a working directory
SRC_DIR = "/teamspace/studios/this_studio/NepaliHTR"
shutil.copytree(cache_path, SRC_DIR, dirs_exist_ok=True)
print(f"Dataset ready at: {SRC_DIR}")

# Verify structure
for sub in ["train", "test"]:
    d = os.path.join(SRC_DIR, sub)
    if os.path.isdir(d):
        xmls = [f for f in os.listdir(d) if f.endswith(".xml")]
        imgs = [f for f in os.listdir(d) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        print(f"  {sub}/  → {len(xmls)} XMLs, {len(imgs)} images")
    else:
        print(f"  WARNING: {sub}/ not found!")

100%|██████████| 1.22G/1.22G [00:08<00:00, 150MB/s] 

Extracting files...


Downloaded to cache: /teamspace/studios/this_studio/.cache/kagglehub/datasets/sweekardahal/nepali-handwritten-images-for-text-detection/versions/2
Dataset ready at: /teamspace/studios/this_studio/NepaliHTR
  train/  → 765 XMLs, 765 images
  test/  → 193 XMLs, 193 images


In [3]:
import xml.etree.ElementTree as ET
import random
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────
DATASET_OUT = "/teamspace/studios/this_studio/nepali_htr_yolo"
CLASS_NAMES = ["text"]
CLASS_MAP = {"text": 0}
VAL_RATIO = 0.2
RANDOM_SEED = 42

print(f"Output dir: {DATASET_OUT}")
print(f"Classes:    {CLASS_NAMES}")
print(f"Val ratio:  {VAL_RATIO}")

Output dir: /teamspace/studios/this_studio/nepali_htr_yolo
Classes:    ['text']
Val ratio:  0.2


In [4]:
# ── VOC → YOLO conversion helpers ─────────────────────────────────────────

def convert_bbox_to_yolo(bbox, img_w, img_h):
    xmin, ymin, xmax, ymax = bbox
    x_center = (xmin + xmax) / 2.0 / img_w
    y_center = (ymin + ymax) / 2.0 / img_h
    w = (xmax - xmin) / img_w
    h = (ymax - ymin) / img_h
    return (
        max(0.0, min(1.0, x_center)),
        max(0.0, min(1.0, y_center)),
        max(0.0, min(1.0, w)),
        max(0.0, min(1.0, h)),
    )


def parse_voc_annotation(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    size = root.find("size")
    img_w = int(size.find("width").text)
    img_h = int(size.find("height").text)

    annotations = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name not in CLASS_MAP:
            continue
        cid = CLASS_MAP[name]
        b = obj.find("bndbox")
        xmin = max(0, min(float(b.find("xmin").text), img_w))
        ymin = max(0, min(float(b.find("ymin").text), img_h))
        xmax = max(0, min(float(b.find("xmax").text), img_w))
        ymax = max(0, min(float(b.find("ymax").text), img_h))
        if xmax <= xmin or ymax <= ymin:
            continue
        xc, yc, w, h = convert_bbox_to_yolo((xmin, ymin, xmax, ymax), img_w, img_h)
        annotations.append(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    return annotations


def find_image(directory, stem):
    for ext in [".jpg", ".jpeg", ".png"]:
        p = os.path.join(directory, stem + ext)
        if os.path.exists(p):
            return p
    return None


print("Conversion functions defined.")

Conversion functions defined.


In [5]:
# ── Build YOLO dataset: split, convert, copy ─────────────────────────────

train_dir = os.path.join(SRC_DIR, "train")
test_dir = os.path.join(SRC_DIR, "test")

train_xmls = sorted([f for f in os.listdir(train_dir) if f.endswith(".xml")])
test_xmls = sorted([f for f in os.listdir(test_dir) if f.endswith(".xml")])

# Split original train → train + val
random.seed(RANDOM_SEED)
random.shuffle(train_xmls)
val_count = int(len(train_xmls) * VAL_RATIO)
val_xmls = train_xmls[:val_count]
final_train_xmls = train_xmls[val_count:]

# Create output directories
output_dir = Path(DATASET_OUT)
if output_dir.exists():
    shutil.rmtree(output_dir)

for split in ["train", "val", "test"]:
    (output_dir / split / "images").mkdir(parents=True)
    (output_dir / split / "labels").mkdir(parents=True)

splits = {
    "train": (final_train_xmls, train_dir),
    "val":   (val_xmls, train_dir),
    "test":  (test_xmls, test_dir),
}

print(f"Train: {len(final_train_xmls)} | Val: {len(val_xmls)} | Test: {len(test_xmls)}")

total_images = 0
total_annotations = 0

for split_name, (xml_list, src_dir) in splits.items():
    skipped = 0
    for xml_file in xml_list:
        stem = os.path.splitext(xml_file)[0]
        xml_path = os.path.join(src_dir, xml_file)

        img_path = find_image(src_dir, stem)
        if img_path is None:
            skipped += 1
            continue

        annotations = parse_voc_annotation(xml_path)

        img_ext = os.path.splitext(img_path)[1]
        shutil.copy2(img_path, output_dir / split_name / "images" / f"{stem}{img_ext}")

        with open(output_dir / split_name / "labels" / f"{stem}.txt", "w") as f:
            f.write("\n".join(annotations))
            if annotations:
                f.write("\n")

        total_images += 1
        total_annotations += len(annotations)

    print(f"  {split_name:5s}: {len(xml_list) - skipped} converted, {skipped} skipped")

print(f"\nTotal: {total_images} images, {total_annotations} annotations")

Train: 612 | Val: 153 | Test: 193
  train: 612 converted, 0 skipped
  val  : 153 converted, 0 skipped
  test : 193 converted, 0 skipped

Total: 958 images, 78159 annotations


In [6]:
# ── Verify & inspect ──────────────────────────────────────────────────────
for split in ["train", "val", "test"]:
    n_imgs = len(os.listdir(output_dir / split / "images"))
    n_lbls = len(os.listdir(output_dir / split / "labels"))
    print(f"{split:5s} → images: {n_imgs}, labels: {n_lbls}")

# Show sample label
sample = sorted(os.listdir(output_dir / "train" / "labels"))[0]
print(f"\nSample label ({sample}):")
with open(output_dir / "train" / "labels" / sample) as f:
    for line in f.readlines()[:5]:
        print(f"  {line.strip()}")

train → images: 612, labels: 612
val   → images: 153, labels: 153
test  → images: 193, labels: 193

Sample label (1.txt):
  0 0.088841 0.038029 0.151679 0.043215
  0 0.291983 0.044944 0.159263 0.044944
  0 0.463164 0.050994 0.141928 0.044944
  0 0.617010 0.052723 0.126761 0.053587
  0 0.785482 0.060501 0.112676 0.041487


In [7]:
# ── Create data.yaml (YOLOv7s compatible) ─────────────────────────────────
import yaml

data_yaml = {
    "path": DATASET_OUT,
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = os.path.join(DATASET_OUT, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f"Saved: {yaml_path}\n")
print(open(yaml_path).read())
print("✅ Dataset ready for YOLOv7s training!")
print(f"   Use: model.train(data='{yaml_path}', ...)")

Saved: /teamspace/studios/this_studio/nepali_htr_yolo/data.yaml

path: /teamspace/studios/this_studio/nepali_htr_yolo
train: train/images
val: val/images
test: test/images
nc: 1
names:
- text

✅ Dataset ready for YOLOv7s training!
   Use: model.train(data='/teamspace/studios/this_studio/nepali_htr_yolo/data.yaml', ...)
